Configure the local data snapshots as described in the README before running this notebook.

# Cross-market information and Ibovespa direction

This notebook reads an archived snapshot through 5 November 2026. It predicts the direction of the next Ibovespa close and evaluates BOVA11 as the execution instrument.

In [ ]:
import sys, subprocess
if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable,"-m","pip","-q","install","exchange-calendars","hmmlearn","arch","lightgbm","xgboost"])

In [ ]:
from retratos_locais import ler_retrato_local

BASES = ler_retrato_local('Ibovespa_Mercados_e_MetaLabeling.ipynb', 'BASES')

# Snapshot da estratégia intradiária, incluindo parâmetros dos estados causais.
INTRADIA_CONGELADO = ler_retrato_local('Ibovespa_Mercados_e_MetaLabeling.ipynb', 'INTRADIA_CONGELADO')


## Prices and exchange calendar

BVMF sessions define the trading calendar. The barrier simulator excludes windows with incomplete execution prices or insufficient remaining sessions.

In [ ]:
from pathlib import Path
import io, base64, zlib, json
import numpy as np
import pandas as pd
from scipy.special import logsumexp
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import brier_score_loss, balanced_accuracy_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from hmmlearn.hmm import GaussianHMM
from arch import arch_model
import exchange_calendars as xcals

SAIDA = Path("resultados_mercados")/"P1_dados"
SAIDA.mkdir(parents=True,exist_ok=True)
SEED = 42
DATA_CAL = pd.Timestamp("2025-01-01")
DATA_VALID = pd.Timestamp("2025-07-01")
DATA_TESTE = pd.Timestamp("2026-01-01")
CUSTO_LADO = .0015  # 10 bps de custos + 5 bps de slippage por lado
PRAZO = 5
FIM_EXPLORATORIO = pd.Timestamp("2026-11-05")
# Opcional. Pasta com CSVs Date/OHLC adicionais, usando os nomes de BASES.
# São acrescentadas apenas datas posteriores à última data de cada série incluída.
PASTA_DADOS_NOVOS = None

def carregar(nome):
    bruto = zlib.decompress(base64.b64decode(BASES[nome]))
    dados = pd.read_csv(io.BytesIO(bruto), index_col="Date", parse_dates=True)
    if PASTA_DADOS_NOVOS is not None:
        arquivo = Path(PASTA_DADOS_NOVOS)/f"{nome}.csv"
        if arquivo.exists():
            novos = pd.read_csv(arquivo,index_col="Date",parse_dates=True)
            dados = pd.concat([dados,novos.loc[novos.index>dados.index.max()]]).sort_index()
    dados.index = dados.index.astype("datetime64[ns]")
    assert dados.index.is_unique and dados.index.is_monotonic_increasing
    return dados

ibov = carregar("ibov")
cal = xcals.get_calendar("BVMF", start="2020-11-01", end=ibov.index.max()+pd.Timedelta(days=15))
sessoes = cal.sessions_in_range(ibov.index.min(), ibov.index.max())
assert sessoes.equals(ibov.index), "O histórico do Ibovespa contém lacunas."
bova = carregar("bova11").reindex(sessoes)
QUARENTENA_BOVA11 = {'2026-10-26': {'motivo': 'OHLC divergente entre provedores. Abertura oficial ainda não confirmada.', 'abertura_yahoo': 140.0, 'abertura_alternativa': 169.0, 'maxima_alternativa': 176.6, 'minima_alternativa': 165.06, 'fechamento_alternativo': 172.42, 'fonte_alternativa': 'https://www.cotab3.com.br/cotacoes?meses=12&ticker=BOVA11', 'referencia_oficial': 'https://www.b3.com.br/pt_br/market-data-e-indices/servicos-de-dados/market-data/historico/mercado-a-vista/cotacoes-historicas/'}}
bova_execucao = bova.copy()
em_quarentena = bova_execucao.index.isin(pd.to_datetime(list(QUARENTENA_BOVA11)))
bova_execucao.loc[em_quarentena,["Open","High","Low","Close"]] = np.nan
print("Sessões em quarentena para execução:",list(bova_execucao.index[em_quarentena].date))
for dados in [ibov, bova]:
    ohlc = dados[["Open","High","Low","Close"]].dropna()
    assert np.isfinite(ohlc).all().all() and (ohlc > 0).all().all()
    assert (ohlc.High >= ohlc.max(axis=1)).all() and (ohlc.Low <= ohlc.min(axis=1)).all()
print("Sessões no Ibovespa:", len(ibov), "| Sem OHLC completo em BOVA11:",
      bova[["Open","High","Low","Close"]].isna().any(axis=1).sum())

## Features available at the forecast origin

The features include momentum, volatility, regimes, and lagged returns for the S&P 500, USD/BRL, VIX, crude oil, and EWZ.

In [ ]:
# O fallback espelha o módulo para execução independente no Colab.
try:
    from features_intradia import construir_features
except ModuleNotFoundError as erro:
    if erro.name != "features_intradia":
        raise
    def construir_features(ibov, sessoes, carregar, congelado):
        """Calcula as variáveis do modelo congelado em ordem causal."""
        r = np.log(ibov.Close).diff()
        X = pd.DataFrame(index=sessoes)
        for k in [0,1,2,3,4]:
            X[f"ret_lag{k}"] = r.shift(k)
        for w in [5,10,21,63]:
            X[f"ret_{w}"] = np.log(ibov.Close).diff(w)
            X[f"vol_{w}"] = r.rolling(w).std()
            X[f"dist_media_{w}"] = np.log(ibov.Close/ibov.Close.rolling(w).mean())
        X["gap"] = np.log(ibov.Open/ibov.Close.shift())
        X["intradia"] = np.log(ibov.Close/ibov.Open)
        X["amplitude"] = np.log(ibov.High/ibov.Low)
        X["dia"] = sessoes.day_of_week
        locais = list(X.columns)

        # Parâmetros estimados nos 252 retornos iniciais e preservados no snapshot.
        estado_hmm = congelado["hmm"]
        mu = np.asarray(estado_hmm["media"])
        var = np.asarray(estado_hmm["variancia"])
        prior = np.asarray(estado_hmm["inicial"])
        transicao = np.asarray(estado_hmm["transicao"])
        regime = []
        for obs in r.dropna().to_numpy()*100:
            logw = np.log(np.maximum(prior,1e-300))-.5*(np.log(var)+(obs-mu)**2/var)
            prior = np.exp(logw-logsumexp(logw)) @ transicao
            regime.append(prior[np.argmax(var)])
        X["hmm_vol_alta"] = pd.Series(regime,index=r.dropna().index)
        estado_garch = congelado["garch"]
        omega, alpha, beta = (estado_garch[k] for k in ["omega","alpha","beta"])
        sigma2 = estado_garch["variancia_inicial"]
        vars_previstas = []
        for obs in r.dropna().to_numpy()*100:
            sigma2 = omega + alpha*obs**2 + beta*sigma2
            vars_previstas.append(sigma2)
        X["garch_vol_prox"] = pd.Series(np.sqrt(vars_previstas)/100,index=r.dropna().index)
        datas_fontes = pd.DataFrame(index=sessoes)

        for nome in ["sp500","dolar","vix","petroleo","ewz"]:
            preco = carregar(nome).Close.dropna()
            if not np.isfinite(preco).all() or not preco.gt(0).all():
                raise ValueError(f"Fechamentos inválidos em {nome}.")
            ext = pd.DataFrame({f"{nome}_ret{w}":np.log(preco).diff(w) for w in [1,5,21]})
            ext["disponivel"] = ext.index
            alinhado = pd.merge_asof(pd.DataFrame(index=sessoes),ext,left_index=True,right_index=True,
                                     allow_exact_matches=False,tolerance=pd.Timedelta(days=7))
            if not (alinhado.disponivel.dropna() < alinhado.disponivel.dropna().index).all():
                raise ValueError(f"Disponibilidade futura em {nome}.")
            datas_fontes[nome] = alinhado.disponivel
            X = X.join(alinhado.drop(columns="disponivel"))

        return r, X, locais, datas_fontes

r, X, locais, datas_fontes = construir_features(ibov, sessoes, carregar, INTRADIA_CONGELADO)
alvo = r.shift(-1)
datas_alvo = pd.Series(sessoes,index=sessoes).shift(-1)
X_origens = X.iloc[253:].dropna()
X = X_origens.loc[alvo.reindex(X_origens.index).notna()]
y = alvo.reindex(X.index).gt(0).astype(int)
datas_alvo = datas_alvo.reindex(X.index)
assert len(X.loc[X.index>=DATA_TESTE]) > 150

def periodo(inicio, fim):
    return X.index[(X.index>=inicio) & (datas_alvo<fim)]

treino = periodo(X.index.min(), DATA_CAL)
calibracao = periodo(DATA_CAL, DATA_VALID)
validacao = periodo(DATA_VALID, DATA_TESTE)
teste = X.index[(X.index>=DATA_TESTE) & (datas_alvo<=FIM_EXPLORATORIO)]
assert datas_alvo.loc[treino].max() < calibracao.min()
assert datas_alvo.loc[calibracao].max() < validacao.min()
assert datas_alvo.loc[validacao].max() < teste.min()
print("Treino, calibração, validação, teste:", *(len(p) for p in [treino,calibracao,validacao,teste]))

## Daily classifier and calibration

Four candidates are compared across four chronological validation windows before 2025. The selective threshold requires at least 20% validation coverage. Test coverage is reported separately.

In [ ]:
def logistica():
    return make_pipeline(StandardScaler(),LogisticRegression(C=.05,random_state=SEED))

candidatos = {
    "Logística OHLC":(logistica(),locais),
    "Logística mercados":(logistica(),list(X.columns)),
    "XGBoost mercados":(XGBClassifier(n_estimators=140,max_depth=2,learning_rate=.025,
        min_child_weight=12,subsample=.8,colsample_bytree=.8,random_state=SEED,n_jobs=2),list(X.columns)),
    "LightGBM mercados":(LGBMClassifier(n_estimators=140,num_leaves=5,min_child_samples=45,
        learning_rate=.025,verbosity=-1,random_state=SEED,n_jobs=2),list(X.columns)),
}
oof = {}
ajustados = {}
for nome,(modelo,cols) in candidatos.items():
    probs = []
    for tr,va in TimeSeriesSplit(n_splits=4,test_size=120,gap=PRAZO).split(treino):
        assert datas_alvo.loc[treino[tr]].max() < treino[va].min()
        ajuste = clone(modelo).fit(X.loc[treino[tr],cols],y.loc[treino[tr]])
        probs.append(pd.Series(ajuste.predict_proba(X.loc[treino[va],cols])[:,1],index=treino[va]))
    oof[nome] = pd.concat(probs)
    ajustados[nome] = clone(modelo).fit(X.loc[treino,cols],y.loc[treino])

def prob(nome, datas):
    return ajustados[nome].predict_proba(X.loc[datas,candidatos[nome][1]])[:,1]

def logits(p):
    p = np.clip(p,1e-5,1-1e-5)
    return np.log(p/(1-p)).reshape(-1,1)

calibradores = {nome:LogisticRegression(C=1).fit(logits(prob(nome,calibracao)),y.loc[calibracao])
               for nome in candidatos}
def prob_cal(nome,datas):
    return calibradores[nome].predict_proba(logits(prob(nome,datas)))[:,1]

ranking = pd.DataFrame([
    {"modelo":nome,"brier_oof":brier_score_loss(y.loc[p.index],p),
     "acuracia_oof":(y.loc[p.index].to_numpy()==(p.to_numpy()>=.5)).mean()}
    for nome,p in oof.items()]).sort_values("brier_oof")
principal = ranking.iloc[0].modelo
p_val = prob_cal(principal,validacao)
regras = []
for corte in [.50,.525,.55,.575,.60,.65,.70]:
    ativo = (p_val>=corte)|(p_val<=1-corte)
    if ativo.sum()>=max(20,.20*len(validacao)):
        regras.append(((y.loc[validacao].to_numpy()[ativo]==(p_val[ativo]>=.5)).mean(),ativo.sum(),corte))
corte_direcao = max(regras,key=lambda linha:linha[:2])[2]
display(ranking)
print("Modelo principal:", principal, "| limiar:", corte_direcao)

## BOVA11 and meta-labeling

The primary classifier sets the trade direction. The secondary filter learns from historical signals generated outside each fitting fold. Simulated entry is at the next open, with a 1.5% profit target, a 1% stop, and a maximum of five sessions.

In [ ]:
try:
    from barreiras_intradia import avaliar_eventos, limite_sem_censura
except ModuleNotFoundError as erro:
    if erro.name!="barreiras_intradia":
        raise
    OHLC_BARREIRAS = ["Open","High","Low","Close"]


    def simular_evento(barras, sessoes, origem, lado, limite, prazo=5, custo_lado=.0015):
        """Lê preços em ordem, após reservar a janela inteira antes do corte."""
        if lado not in (-1,1) or prazo<1 or not np.isfinite(custo_lado) or custo_lado<0:
            raise ValueError("Lado, prazo ou custo inválido.")
        inicio = sessoes.get_loc(origem)+1
        janela = sessoes[inicio:inicio+prazo]
        evento = {"origem":origem,"lado":lado,
                  "entrada_data":janela[0] if len(janela) else pd.NaT,
                  "vencimento":janela[-1] if len(janela)==prazo else pd.NaT}
        if len(janela)!=prazo or janela[-1]>=limite:
            return evento|{"status":"fora_da_janela","motivo":"prazo_cruza_o_corte"}
        marcas = {}
        for pos,data in enumerate(janela):
            if data not in barras.index:
                valores = np.full(4,np.nan)
            else:
                valores = barras.loc[data,OHLC_BARREIRAS].to_numpy(dtype=float)
            abertura,maxima,minima,fechamento = valores
            if (not np.isfinite(valores).all() or (valores<=0).any() or
                maxima<valores.max() or minima>valores.min()):
                return evento|{"status":"censurado" if pos else "entrada_indisponivel",
                               "motivo":"ohlc_ausente_ou_invalido","data_interrupcao":data}
            if pos==0:
                entrada = abertura
                alvo = entrada*(1+lado*.015)
                stop = entrada*(1-lado*.01)
            pior = minima if lado==1 else maxima
            melhor = maxima if lado==1 else minima
            if lado*(abertura-stop)<=0:
                preco,motivo = abertura,"stop_abertura"
            elif lado*(abertura-alvo)>=0:
                preco,motivo = alvo,"alvo_abertura"
            elif lado*(pior-stop)<=0:
                preco,motivo = stop,"stop"
            elif lado*(melhor-alvo)>=0:
                preco,motivo = alvo,"alvo"
            elif pos==prazo-1:
                preco,motivo = fechamento,"prazo"
            else:
                marcas[data] = lado*(fechamento/entrada-1)-custo_lado
                continue
            liquido = lado*(preco/entrada-1)-2*custo_lado
            marcas[data] = liquido
            return evento|{"status":"encerrado","motivo":motivo,"saida":data,
                           "liquido":liquido,"marcas":marcas}
        raise RuntimeError("A janela elegível deveria terminar em uma saída ou censura.")


    def avaliar_eventos(barras,sessoes,probabilidades,limite,prazo=5,custo_lado=.0015):
        """Devolve rótulos conhecidos e auditoria de todas as oportunidades."""
        if not probabilidades.index.is_unique or not np.isfinite(probabilidades).all():
            raise ValueError("As oportunidades exigem origens únicas e valores finitos.")
        encerrados, auditoria = {}, []
        for origem,valor in probabilidades.sort_index().items():
            evento = simular_evento(barras,sessoes,origem,1 if valor>=.5 else -1,
                                    limite,prazo,custo_lado)
            if evento["status"]=="encerrado":
                encerrados[origem] = evento
            auditoria.append({k:v for k,v in evento.items() if k!="marcas"})
        return encerrados,pd.DataFrame(auditoria,columns=["origem","lado","entrada_data",
            "vencimento","status","motivo","saida","liquido","data_interrupcao"])


    def limite_sem_censura(barras, indice, limite):
        """Encerra o painel antes da primeira sessão com OHLC não confiável."""
        ohlc = barras.reindex(indice)[OHLC_BARREIRAS]
        valido = (np.isfinite(ohlc).all(axis=1)&ohlc.gt(0).all(axis=1)&
                  ohlc.High.ge(ohlc.max(axis=1))&ohlc.Low.le(ohlc.min(axis=1)))
        faltantes = ohlc.index[~valido]
        return min(limite,faltantes[0]) if len(faltantes) else limite

sessoes_barreiras = cal.sessions_in_range(sessoes[0],sessoes[-1]+pd.Timedelta(days=10))

def eventos(p,limite,etiqueta):
    registros,auditoria = avaliar_eventos(bova_execucao,sessoes_barreiras,p,limite,PRAZO,CUSTO_LADO)
    auditoria.assign(corte_exclusivo=limite).to_csv(
        SAIDA/f"audit_barreiras_{etiqueta}.csv",index=False)
    return registros

def features_meta(p, registros):
    idx = pd.DatetimeIndex(list(registros))
    x = X.loc[idx].copy()
    x["lado"] = np.where(p.loc[idx]>=.5,1,-1)
    x["forca"] = (p.loc[idx]-.5).abs()
    return x

p_oof = oof[principal]
ev_treino = eventos(p_oof,DATA_CAL,"direcional_treino")
xm = features_meta(p_oof,ev_treino)
ym = [int(e["liquido"]>0) for e in ev_treino.values()]
meta = logistica().fit(xm,ym)

def simular(registros, aceitos, indice):
    if len(indice)==0:
        return {"operacoes":0,"acerto":np.nan,"retorno_liquido":np.nan,
                "profit_factor":np.nan,"drawdown":np.nan,"sharpe":np.nan},pd.Series(dtype=float)
    capital = 1.
    ultima_saida = pd.Timestamp.min
    curva = pd.Series(np.nan,index=indice,dtype=float)
    trades = []
    for data,e in registros.items():
        if data < ultima_saida or not aceitos.loc[data]:
            continue
        for dia,marca in e["marcas"].items():
            curva.loc[dia] = capital*(1+marca)
        capital *= 1+e["liquido"]
        ultima_saida = e["saida"]
        trades.append(e["liquido"])
    curva = curva.ffill().fillna(1.)
    ganhos = sum(v for v in trades if v>0)
    perdas = -sum(v for v in trades if v<0)
    rd = curva.pct_change().fillna(curva.iloc[0]-1)
    pico = curva.cummax().clip(lower=1.)
    return {"operacoes":len(trades),"acerto":np.mean(np.array(trades)>0) if trades else np.nan,
            "retorno_liquido":capital-1,"profit_factor":ganhos/perdas if perdas else np.nan,
            "drawdown":(curva/pico-1).min(),
            "sharpe":np.sqrt(252)*rd.mean()/rd.std() if rd.std()>0 else np.nan},curva

# Usa pontuação bruta da mesma família que gerou os sinais OOF para treinar o filtro.
p_meta_val = pd.Series(prob(principal,validacao),index=validacao)
cal_val = sessoes[(sessoes>=DATA_VALID)&(sessoes<DATA_TESTE)]
fim_val_barreiras = limite_sem_censura(bova_execucao,cal_val,DATA_TESTE)
cal_val_barreiras = cal_val[cal_val<fim_val_barreiras]
ev_val = eventos(p_meta_val,fim_val_barreiras,"direcional_validacao")
xv = features_meta(p_meta_val,ev_val)
mv = pd.Series(meta.predict_proba(xv)[:,1],index=xv.index)
opcoes = []
for limiar in [.45,.50,.55,.60,.65,.70]:
    met,_ = simular(ev_val,mv>=limiar,cal_val_barreiras)
    if met["operacoes"]>=12:
        opcoes.append((met["retorno_liquido"],-limiar))
corte_meta = -max(opcoes)[1] if opcoes else None
(SAIDA/"regras_congeladas.json").write_text(json.dumps({
    "principal":principal,"corte_direcao":float(corte_direcao),"corte_meta":corte_meta,
    "fim_selecao":"2025-12-31","custo_lado":CUSTO_LADO},indent=2))
print("Limiar do meta-modelo:",corte_meta)

## Directional evaluation in 2026

The tables report accuracy, coverage, and a majority-class benchmark on the same dates.

In [ ]:
p = prob_cal(principal,teste)
real = y.loc[teste].to_numpy()
maioria = int(y.loc[treino].mean()>=.5)
linhas = []
comparacoes = [
    ("Classe majoritária",np.full(len(teste),maioria),np.ones(len(teste),bool)),
    ("Persistência",r.loc[teste].gt(0).to_numpy(),np.ones(len(teste),bool)),
    (principal+" seletivo",p>=.5,(p>=corte_direcao)|(p<=1-corte_direcao))]
comparacoes.extend((nome,prob_cal(nome,teste)>=.5,np.ones(len(teste),bool))
                   for nome in candidatos)
for nome, pred, ativo in comparacoes:
    linhas.append({"modelo":nome,"acertos":int((pred[ativo]==real[ativo]).sum()),
        "sinais":int(ativo.sum()),"cobertura":ativo.mean(),
        "acuracia":(pred[ativo]==real[ativo]).mean() if ativo.any() else np.nan,
        "acuracia_balanceada":balanced_accuracy_score(real[ativo],pred[ativo]) if len(np.unique(real[ativo]))==2 else np.nan,
        "referencia_mesmos_dias":(real[ativo]==maioria).mean() if ativo.any() else np.nan})
direcao = pd.DataFrame(linhas)
print("Alvos de teste:",datas_alvo.loc[teste].min().date(),"a",datas_alvo.loc[teste].max().date())
print("Brier bruto e calibrado:", brier_score_loss(real,prob(principal,teste)), brier_score_loss(real,p))
display(direcao)
direcao.to_csv(SAIDA/"direcao_2026.csv",index=False)
pd.DataFrame({"data_alvo":datas_alvo.loc[teste],"alta_real":real,"prob_alta":p,
              "emitiu_sinal":(p>=corte_direcao)|(p<=1-corte_direcao)}).to_csv(SAIDA/"previsoes_2026.csv")

## Trading evaluation in 2026

Compare the primary signal with and without a filter under the same execution assumptions and costs. The cash position is a separate benchmark.

In [ ]:
p_meta = pd.Series(prob(principal,teste),index=teste)
idx_teste = sessoes[(sessoes>=DATA_TESTE)&(sessoes<=FIM_EXPLORATORIO)]
fim_teste_barreiras = limite_sem_censura(bova_execucao,idx_teste,FIM_EXPLORATORIO+pd.Timedelta(days=1))
idx_barreiras = idx_teste[idx_teste<fim_teste_barreiras]
ev_teste = eventos(p_meta,fim_teste_barreiras,"direcional_teste")
xt = features_meta(p_meta,ev_teste)
mt = pd.Series(meta.predict_proba(xt)[:,1] if len(xt) else [],index=xt.index,dtype=float)
filtros = {"Sem filtro":pd.Series(True,index=xt.index),"Caixa":pd.Series(False,index=xt.index)}
if corte_meta is not None:
    filtros["Meta-labeling"] = mt>=corte_meta
metricas = {}
curvas = {}
for nome,aceitos in filtros.items():
    metricas[nome],curvas[nome] = simular(ev_teste,aceitos,idx_barreiras)
trade = pd.DataFrame(metricas).T
trade["pregoes_avaliados"] = len(idx_barreiras)
display(trade)
trade.to_csv(SAIDA/"operacoes_2026.csv")
pd.DataFrame(curvas).to_csv(SAIDA/"curvas_2026.csv")
ax = pd.DataFrame(curvas).plot(title="BOVA11 em 2026 · simulação após custos",ylabel="Capital por unidade inicial")
ax.figure.tight_layout()
ax.figure.savefig(SAIDA/"curvas_2026.png",dpi=150)

## Earlier version

Archived results preceding the unified barrier rules are retained in the original research workspace. The cells above produce the current historical evaluation.

## Persistence diagnostics

The directional forecast targets the next Ibovespa close. Barrier trades enter BOVA11 at the following open and use a maximum holding period of five sessions. The diagnostics break results down by the available groups.

In [ ]:
from arch.bootstrap import CircularBlockBootstrap
from xgboost import XGBRegressor

persistencia = r.reindex(X.index).gt(0).astype(int)
lim_vol = float(X.loc[treino,"garch_vol_prox"].median())
estados = pd.DataFrame({
    "volatilidade": np.where(X.garch_vol_prox>=lim_vol,"Alta","Baixa"),
    "regime_hmm": np.where(X.hmm_vol_alta>=.5,"Alta","Baixa")},index=X.index)
z = X[["hmm_vol_alta","garch_vol_prox"]].copy()
z["direcao_anterior"] = 2*persistencia-1
z["direcao_x_regime"] = z.direcao_anterior*z.hmm_vol_alta
modelo_regime = logistica().fit(z.loc[treino],y.loc[treino])

ret_norm = alvo.reindex(X.index)/X.garch_vol_prox
def regressor():
    return XGBRegressor(n_estimators=140,max_depth=2,learning_rate=.025,min_child_weight=12,
                        subsample=.8,colsample_bytree=.8,random_state=SEED,n_jobs=2)
modelo_normalizado = regressor().fit(X.loc[treino],ret_norm.loc[treino])

def diagnosticar_direcao(datas, etiqueta):
    reais = y.loc[datas]
    previsoes = pd.DataFrame({
        "Persistência":persistencia.loc[datas],
        "Classe majoritária":int(y.loc[treino].mean()>=.5),
        "Modelo anterior":(prob_cal(principal,datas)>=.5).astype(int),
        "HMM + persistência":modelo_regime.predict(z.loc[datas]),
        "XGBoost retorno normalizado":(modelo_normalizado.predict(X.loc[datas])>0).astype(int)
    },index=datas)
    acertos = previsoes.eq(reais,axis=0)
    print(etiqueta, "|", len(datas), "previsões")
    display(pd.DataFrame({"acertos":acertos.sum(),"acuracia":acertos.mean()}))
    por_estado = []
    for campo in estados:
        tabela = acertos.assign(grupo=estados.loc[datas,campo]).groupby("grupo")
        contagens = tabela.size()
        resultado = tabela.mean().assign(dias=contagens,segmentacao=campo)
        por_estado.append(resultado.reset_index())
    display(pd.concat(por_estado,ignore_index=True))
    pd.concat(por_estado,ignore_index=True).to_csv(SAIDA/f"regimes_{etiqueta}.csv",index=False)
    linhas = []
    if len(datas)>=40:
        for nome in acertos.columns:
            dif = acertos[nome].astype(float)-acertos["Persistência"].astype(float)
            for bloco in [5,10,20]:
                bs = CircularBlockBootstrap(bloco,dif.to_numpy(),seed=SEED)
                ic = bs.conf_int(np.mean,reps=3000,method="percentile")
                linhas.append({"modelo":nome,"bloco":bloco,"dif_pp":100*dif.mean(),
                               "IC_inf_pp":100*ic[0,0],"IC_sup_pp":100*ic[1,0]})
        intervalos = pd.DataFrame(linhas)
        display(intervalos.loc[intervalos.bloco.eq(10)])
        intervalos.to_csv(SAIDA/f"intervalos_pareados_{etiqueta}.csv",index=False)
    return acertos

# Histórico pré-2026 apenas descritivo para a regra fixa de persistência.
janelas = {"validação_2025":validacao,"exploratorio_2026":teste}
display(pd.DataFrame([{"janela":nome,"n":len(d),
    "acuracia_persistencia":persistencia.loc[d].eq(y.loc[d]).mean()} for nome,d in janelas.items()]))

## Persistence filter and expected net return

The filter uses the persistence direction and net barrier outcomes. The threshold is selected in the second half of 2025 with a minimum of 12 trades.

In [ ]:
def previsao_persistencia(datas):
    return persistencia.loc[datas].astype(float)

ev_persist_treino = eventos(previsao_persistencia(treino),DATA_CAL,"persistencia_treino")
xp = features_meta(previsao_persistencia(treino),ev_persist_treino)
modelo_filtro_persist = logistica().fit(xp,[int(e["liquido"]>0) for e in ev_persist_treino.values()])

def pontuar_persistencia(datas, fim, etiqueta):
    primario = previsao_persistencia(datas)
    ev = eventos(primario,fim,etiqueta)
    if not ev:
        return ev,pd.Series(dtype=float)
    xm = features_meta(primario,ev)
    return ev,pd.Series(modelo_filtro_persist.predict_proba(xm)[:,1],index=xm.index)

ev_pv, pontuacao_pv = pontuar_persistencia(validacao,fim_val_barreiras,"persistencia_validacao")
escolhas = []
for corte in [.45,.50,.55,.60,.65,.70]:
    met,_ = simular(ev_pv,pontuacao_pv>=corte,cal_val_barreiras)
    if met["operacoes"]>=12:
        escolhas.append((met["retorno_liquido"],-corte))
corte_persist = -max(escolhas)[1] if escolhas else None

ret_bova = (bova_execucao.Close/bova_execucao.Open-1).shift(-1).reindex(X.index)
import hashlib
modelo_bytes = base64.b64decode(INTRADIA_CONGELADO["modelo_ubj"])
assert hashlib.sha256(modelo_bytes).hexdigest() == INTRADIA_CONGELADO["sha256"]
assert list(X.columns) == INTRADIA_CONGELADO["features"]
assert CUSTO_LADO == INTRADIA_CONGELADO["custo_lado"]
modelo_ret = XGBRegressor()
modelo_ret.load_model(bytearray(modelo_bytes))
margem = INTRADIA_CONGELADO["margem"]
(SAIDA/"modelo_intradia_congelado.ubj").write_bytes(modelo_bytes)
manifesto = {k:v for k,v in INTRADIA_CONGELADO.items() if k!="modelo_ubj"}
manifesto["primeira_origem_confirmacao"] = str((FIM_EXPLORATORIO+pd.Timedelta(days=1)).date())
(SAIDA/"manifesto_intradia_congelado.json").write_text(json.dumps(manifesto,indent=2),encoding="utf-8")
(SAIDA/"regras_diagnostico.json").write_text(json.dumps({
    "treino_ate":"2024-12-31","validacao_ate":"2025-12-31",
    "fim_exploratorio":str(FIM_EXPLORATORIO.date()),"lim_vol_garch":lim_vol,
    "corte_hmm":.5,"corte_filtro_persistencia":corte_persist,
    "custo_lado":CUSTO_LADO,"margem_retorno_esperado":margem,
    "blocos_bootstrap":[5,10,20],"seed":SEED},indent=2),encoding="utf-8")

def medir_operacoes(registros, aceitos, indice):
    executadas = {}
    ultima = pd.Timestamp.min
    for origem,e in registros.items():
        if origem>=ultima and aceitos.loc[origem]:
            executadas[origem] = e
            ultima = e["saida"]
    met,curva = simular(executadas,pd.Series(True,index=list(executadas)),indice)
    liquidos = np.array([e["liquido"] for e in executadas.values()])
    ganhos,perdas = liquidos[liquidos>0],-liquidos[liquidos<0]
    met["acertos_equilibrio"] = perdas.mean()/(ganhos.mean()+perdas.mean()) if len(ganhos) and len(perdas) else np.nan
    met["exposicao"] = sum(len(e["marcas"]) for e in executadas.values())/len(indice) if len(indice) else np.nan
    met["turnover_por_100_pregoes"] = 200*len(executadas)/len(indice) if len(indice) else np.nan
    return met,curva,executadas

def avaliar_trades(datas, indice, fim, etiqueta):
    fim_barreiras = limite_sem_censura(bova_execucao,indice,fim)
    indice_barreiras = indice[indice<fim_barreiras]
    ev, pontuacao = pontuar_persistencia(datas,fim_barreiras,f"persistencia_{etiqueta}")
    regras = {"Persistência com barreiras":(ev,pd.Series(True,index=pontuacao.index))}
    if corte_persist is not None:
        regras["Persistência + meta-labeling"] = (ev,pontuacao>=corte_persist)
    previsto = pd.Series(modelo_ret.predict(X.loc[datas])*X.loc[datas,"garch_vol_prox"],index=datas)
    intradia = {}
    for origem,v in previsto.items():
        data = datas_alvo.loc[origem]
        ret = ret_bova.loc[origem]
        if pd.notna(ret) and data<fim and abs(v)>2*CUSTO_LADO+margem:
            liquido = np.sign(v)*ret-2*CUSTO_LADO
            intradia[origem] = {"saida":data,"liquido":liquido,"marcas":{data:liquido}}
    regras["Retorno esperado intradiário"] = (intradia,pd.Series(True,index=list(intradia)))
    metricas,curvas,executadas = {},{},{}
    for nome,(eventos_regra,aceitos) in regras.items():
        metricas[nome],curvas[nome],executadas[nome] = medir_operacoes(
            eventos_regra,aceitos,indice if nome=="Retorno esperado intradiário" else indice_barreiras)
        metricas[nome]["pregoes_avaliados"] = len(indice if nome=="Retorno esperado intradiário" else indice_barreiras)
    tab = pd.DataFrame(metricas).T
    display(tab)
    tab.to_csv(SAIDA/f"trades_diagnostico_{etiqueta}.csv")
    pd.DataFrame(curvas).to_csv(SAIDA/f"curvas_diagnostico_{etiqueta}.csv")
    pd.DataFrame(curvas).plot(title=f"BOVA11 · {etiqueta}",ylabel="Capital por unidade inicial")
    filtradas = executadas.get("Persistência + meta-labeling",{})
    if filtradas:
        # Controle retrospectivo pareado por duração, sem usar retorno na amostragem.
        base = list(executadas["Persistência com barreiras"].values())
        grupos = {}
        for duracao in range(1,PRAZO+1):
            n_alvo = sum(len(e["marcas"])==duracao for e in filtradas.values())
            pool = np.array([e["liquido"] for e in base if len(e["marcas"])==duracao])
            if n_alvo:
                grupos[duracao] = (n_alvo,pool)
        if all(n<=len(pool) for n,pool in grupos.values()):
            rng = np.random.default_rng(SEED)
            retorno_sorteado = np.ones(2000)
            for n,pool in grupos.values():
                retorno_sorteado *= [np.prod(1+rng.choice(pool,size=n,replace=False))
                                     for _ in range(len(retorno_sorteado))]
            q = np.quantile(retorno_sorteado-1,[.025,.5,.975])
            controle = pd.DataFrame([{"operacoes":len(filtradas),"q025":q[0],"mediana":q[1],
                                      "q975":q[2],"retorno_filtro":tab.loc["Persistência + meta-labeling","retorno_liquido"]}])
            display(controle)
            controle.to_csv(SAIDA/f"controle_exposicao_{etiqueta}.csv",index=False)
        else:
            print("A referência sem filtro não tem operações suficientes nas mesmas durações.")
    return tab

## Exploratory evaluation and future protocol

Confidence bins use calibrated class probabilities. Trading summaries include exposure, turnover, and the break-even hit rate. The historical exercise is exploratory, and prospective signals follow a separate protocol.

In [ ]:
acertos_expl = diagnosticar_direcao(teste,"exploratorio_2026")
prob_expl = prob_cal(principal,teste)
confianca = np.maximum(prob_expl,1-prob_expl)
faixas = pd.DataFrame({"faixa":pd.cut(confianca,[.5,.55,.6,.65,.7,1.],include_lowest=True),
    "confianca":confianca,"acerto":acertos_expl["Modelo anterior"].to_numpy()})
calib = faixas.groupby("faixa",observed=False).agg(
    sinais=("acerto","size"),confianca_media=("confianca","mean"),acuracia=("acerto","mean"))
calib["cobertura"] = calib.sinais/len(teste)
display(calib)
calib.to_csv(SAIDA/"calibracao_faixas_2026.csv")
trades_expl = avaliar_trades(teste,idx_teste,FIM_EXPLORATORIO+pd.Timedelta(days=1),"exploratorio_2026")
confirmacao = X.index[X.index>FIM_EXPLORATORIO]
if len(confirmacao):
    acertos_confirmacao = diagnosticar_direcao(confirmacao,"confirmacao")
    avaliar_trades(confirmacao,sessoes[sessoes>FIM_EXPLORATORIO],
                   sessoes[-1]+pd.Timedelta(days=1),"confirmacao")
else:
    print("O retrato incluído termina em 05/11/2026. A avaliação posterior aguarda novos pregões.")

## Earlier persistence diagnostics

An earlier version recorded 116 correct persistence predictions out of 210. The current analysis applies the unified barrier and execution rules.

## Audit of the frozen intraday signal

The local snapshot supplies the XGBoost model trained through 2024, its HMM and GARCH parameters, and the feature order. A hash identifies the model. The decision margin is 0.05% above estimated round-trip costs.

In [ ]:
from statsmodels.stats.proportion import proportion_confint

def auditar_intradia(datas, etiqueta):
    origens = pd.DatetimeIndex(datas)
    execucoes = pd.DatetimeIndex(datas_alvo.loc[origens])
    assert execucoes.equals(sessoes[sessoes.get_indexer(origens)+1])
    fontes = datas_fontes.loc[origens].add_prefix("fonte_")
    for coluna in fontes:
        assert fontes[coluna].notna().all()
        assert (fontes[coluna] < origens).all()
    barras = bova_execucao.loc[execucoes,["Open","Close"]].set_axis(origens)
    previsao = modelo_ret.predict(X.loc[origens])*X.loc[origens,"garch_vol_prox"]
    elegivel = barras.notna().all(axis=1)
    sinal = previsao.abs()>2*CUSTO_LADO+margem
    log = pd.DataFrame({"execucao":execucoes,"previsao_bruta":previsao,
        "elegivel":elegivel,"sinal":sinal,"abertura":barras.Open,"fechamento":barras.Close,
        "volatilidade":estados.loc[origens,"volatilidade"],
        "tendencia_21":np.where(X.loc[origens,"ret_21"]>=0,"Positiva","Negativa")}).join(fontes)
    log.index.name = "origem"
    trades = log.loc[elegivel&sinal].copy()
    trades["lado"] = np.sign(trades.previsao_bruta).astype(int)
    trades["bruto"] = trades.lado*(trades.fechamento/trades.abertura-1)
    trades["custo_total"] = 2*CUSTO_LADO
    trades["liquido"] = trades.bruto-trades.custo_total
    log.to_csv(SAIDA/f"auditoria_origens_{etiqueta}.csv")
    trades.to_csv(SAIDA/f"auditoria_operacoes_{etiqueta}.csv")
    if trades.empty:
        print(etiqueta, "| nenhuma operação com desfecho disponível")
        return None
    n = len(trades)
    acertos = int(trades.liquido.gt(0).sum())
    lo,hi = proportion_confint(acertos,n,method="wilson")
    retorno = float((1+trades.liquido).prod()-1)
    media = float(trades.liquido.mean())
    elegiveis = log.loc[elegivel]
    pool = (elegiveis.fechamento/elegiveis.abertura-1).to_numpy()
    rng = np.random.default_rng(SEED)
    sorteios = np.array([rng.choice(len(pool),size=n,replace=False) for _ in range(10000)])
    pnl = pool[sorteios]*trades.lado.to_numpy()-2*CUSTO_LADO
    controle = np.prod(1+pnl,axis=1)-1
    q = np.quantile(controle,[.025,.5,.975])
    resumo = pd.DataFrame([{"origens":len(log),"elegiveis":int(elegivel.sum()),
        "sinais_sem_preco":int((sinal&~elegivel).sum()),"operacoes":n,
        "compradas":int(trades.lado.eq(1).sum()),"vendidas":int(trades.lado.eq(-1).sum()),
        "acertos":acertos,"precision":acertos/n,"wilson_inf":lo,"wilson_sup":hi,
        "expectancy_liquida":media,"retorno_liquido":retorno,
        "controle_q025":q[0],"controle_mediana":q[1],"controle_q975":q[2],
        "percentil_vs_controle":(controle<=retorno).mean()}])
    display(resumo.T)
    resumo.to_csv(SAIDA/f"auditoria_resumo_{etiqueta}.csv",index=False)
    grupos = []
    for coluna in ["volatilidade","tendencia_21"]:
        for grupo, candidatas in log.loc[elegivel].groupby(coluna):
            ativas = trades.loc[trades.index.intersection(candidatas.index),"liquido"]
            grupos.append({"eixo":coluna,"grupo":grupo,"elegiveis":len(candidatas),
                "operacoes":len(ativas),"acertos":int(ativas.gt(0).sum()),
                "cobertura":len(ativas)/len(candidatas),"expectancy_liquida":ativas.mean()})
    segmentos = pd.DataFrame(grupos)
    display(segmentos)
    segmentos.to_csv(SAIDA/f"auditoria_regimes_{etiqueta}.csv",index=False)
    cenarios = []
    for slip_bps in [0,5,10,20]:
        liquidos = trades.bruto-2*(10+slip_bps)/10000
        cenarios.append({"slippage_bps_por_lado":slip_bps,"operacoes":n,
            "acertos":int(liquidos.gt(0).sum()),"expectancy_liquida":liquidos.mean(),
            "retorno_liquido":(1+liquidos).prod()-1})
    sensibilidade = pd.DataFrame(cenarios)
    display(sensibilidade)
    sensibilidade.to_csv(SAIDA/f"auditoria_slippage_{etiqueta}.csv",index=False)
    return resumo

historico_auditado = auditar_intradia(teste,"exploratorio_2026")
anterior = trades_expl.loc["Retorno esperado intradiário"]
assert historico_auditado.iloc[0].operacoes == anterior.operacoes
assert np.isclose(historico_auditado.iloc[0].retorno_liquido,anterior.retorno_liquido)
if len(confirmacao):
    auditar_intradia(confirmacao,"confirmacao")
else:
    print("A confirmação aguarda dados posteriores a 05/11/2026.")
